# 16S rPCA Analyses of the Longitudinal Acne Study

Date created: 10/18/2024

Notebook author: Britta De Pessemier

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- rPCA beta diversity analysis for 16S data

In [247]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
from skbio.stats.ordination import OrdinationResults
import biom
from biom import load_table
from gemelli.rpca import rpca
from matplotlib.patches import Ellipse
from skbio.stats.distance import permanova, DistanceMatrix
from itertools import combinations
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
import os

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

# Set display precision globally
pd.set_option('display.float_format', '{:.10f}'.format)

In [248]:
# ----------------------------------------------------------------------------
# Copyright (c) 2016-2023, QIIME 2 development team.
#
# Distributed under the terms of the Modified BSD License.
#
# The full license is in the file LICENSE, distributed with this software.
# ----------------------------------------------------------------------------

# Note: Functions are from https://github.com/qiime2/q2-feature-table/blob/dev/q2_feature_table/_filter.py; an alternative to direct import from my qiime2 environment
def _validate_nonempty_table(table):
    if table.is_empty():
        raise ValueError("The resulting table is empty. This can happen if "
                         "you filter all samples or features out of the "
                         "table. Please check your filtering parameters and "
                         "try again.")


def _get_biom_filter_function(ids_to_keep, min_frequency, max_frequency,
                              min_nonzero, max_nonzero):
    ids_to_keep = set(ids_to_keep)
    if max_frequency is None:
        max_frequency = np.inf
    if max_nonzero is None:
        max_nonzero = np.inf

    def f(data_vector, id_, metadata):
        return (id_ in ids_to_keep) and \
               (min_frequency <= data_vector.sum() <= max_frequency) and \
               (min_nonzero <= (data_vector > 0).sum() <= max_nonzero)
    return f


_other_axis_map = {'sample': 'observation', 'observation': 'sample'}


def _filter_table(table, min_frequency, max_frequency, min_nonzero,
                  max_nonzero, metadata, where, axis, exclude_ids=False,
                  filter_opposite_axis=True,
                  allow_empty_table=False):
    if min_frequency == 0 and max_frequency is None and min_nonzero == 0 and\
       max_nonzero is None and metadata is None and where is None and\
       exclude_ids is False:
        raise ValueError("No filtering was requested.")
    if metadata is None and where is not None:
        raise ValueError("Metadata must be provided if 'where' is "
                         "specified.")
    if metadata is None and exclude_ids is True:
        raise ValueError("Metadata must be provided if 'exclude_ids' "
                         "is True.")
    if metadata is not None:
        ids_to_keep = metadata.get_ids(where=where)
    else:
        ids_to_keep = table.ids(axis=axis)
    if exclude_ids is True:
        ids_to_keep = set(table.ids(axis=axis)) - set(ids_to_keep)

    filter_fn1 = _get_biom_filter_function(
        ids_to_keep, min_frequency, max_frequency, min_nonzero, max_nonzero)
    table.filter(filter_fn1, axis=axis, inplace=True)

    # filter on the opposite axis to remove any entities that now have a
    # frequency of zero
    if filter_opposite_axis:
        table.remove_empty(axis=_other_axis_map[axis], inplace=True)

    if not allow_empty_table:
        _validate_nonempty_table(table)


def filter_samples(table: biom.Table, min_frequency: int = 0,
                   max_frequency: int = None, min_features: int = 0,
                   max_features: int = None,
                   metadata: q2.Metadata = None, where: str = None,
                   exclude_ids: bool = False,
                   filter_empty_features: bool = True,
                   allow_empty_table: bool = False)\
                  -> biom.Table:
    _filter_table(table=table, min_frequency=min_frequency,
                  max_frequency=max_frequency, min_nonzero=min_features,
                  max_nonzero=max_features, metadata=metadata,
                  where=where, axis='sample', exclude_ids=exclude_ids,
                  filter_opposite_axis=filter_empty_features,
                  allow_empty_table=allow_empty_table
                  )
    return table

In [249]:
# Load in tables and metadata
v3_table = Artifact.load('../Data/16S/Tables/16S_V1-V3_rarefied_uncollapsed_table.qza')
v4_table = Artifact.load('../Data/16S/Tables/16S_V4_rarefied_uncollapsed_table.qza')
metadata = Metadata.load('../Metadata/57610_57610_analysis_mapping.txt')
taxonomy = Metadata.load('../Metadata/174116_taxonomy.tsv')

# Convert qiime2 table artifacts to biom
v3_table = v3_table.view(biom.Table)
v4_table = v4_table.view(biom.Table)

# Run filter_samples using q2 filter_samples method
v3_table_filt = filter_samples(table=v3_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)
v4_table_filt = filter_samples(table=v4_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)

In [250]:
# Read in metadata and preprocess
md = pd.read_csv('../Metadata/57610_57610_analysis_mapping.txt', sep='\t', index_col=0)
md = md.loc[md.sample_type=='skin']
md['acne_severity']= md[['visual_assessment_on_photography_acne_severity_cheek_left', 'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
md['inflammatory_lesions_face']= md[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
md['noninflammatory_lesions_face']= md[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
md['a']= md[['skincam_a_cheek_left','skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

md.loc[md['subject_randomization_number'].astype(int)>299, 'cohort'] = 'acne'
md.loc[md['subject_randomization_number'].astype(int)<299, 'cohort'] = 'control'

md['subject_randomization_id'] = md['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)

md['class'] = md['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')

# Define the conditions for assigning values to the "category" column
condition1 = (md['c_zone'] == 'C3')
condition2 = (md['c_zone'].isin(['C1', 'C2']) & (md['cohort'] == 'acne'))
condition3 = (md['c_zone'].isin(['C1', 'C2']) & (md['cohort'] == 'control'))

# Use the conditions to assign values to the "category" column
md.loc[condition1, 'category'] = 'clear_zone'
md.loc[condition2, 'category'] = 'acne'
md.loc[condition3, 'category'] = 'control'

features_of_interest = ['a','skin_ph_meter_ph_mean_forehead_right', 'sebumeter_casual_level_mean_forehead_left',
                    'acne_severity', 'inflammatory_lesions_face', 'noninflammatory_lesions_face','day']#,'subject_randomization_id','subject_randomization_number', 'class', 'cohort', 'Current_Natural_Log_Ratio']

md = md.astype({col: 'str' for col in md.columns[md.dtypes == 'bool']})

In [251]:
# Preprocess metadata
md = md.loc[md.sample_type == 'skin']

# Calculate severity-related columns
md['local_lesion_severity'] = md[['visual_assessment_on_photography_acne_severity_cheek_left', 
                                  'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
md['acne_severity'] = md[['visual_assessment_on_photography_acne_severity_cheek_left', 
                          'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
md['inflammatory_lesions_face'] = md[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
md['noninflammatory_lesions_face'] = md[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
md['a'] = md[['skincam_a_cheek_left', 'skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

# Define cohorts and other columns
md.loc[md['subject_randomization_number'].astype(int) > 299, 'cohort'] = 'acne'
md.loc[md['subject_randomization_number'].astype(int) < 299, 'cohort'] = 'control'
md['subject_randomization_id'] = md['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)
md['class'] = md['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')

In [252]:
# Define conditions for "category" column
condition1 = (md['c_zone'] == 'C3')
condition2 = (md['c_zone'].isin(['C1', 'C2']) & (md['cohort'] == 'acne'))
condition3 = (md['c_zone'].isin(['C1', 'C2']) & (md['cohort'] == 'control'))

# Assign values to "category" based on conditions
md.loc[condition1, 'category'] = 'clear_zone'
md.loc[condition2, 'category'] = 'acne'
md.loc[condition3, 'category'] = 'control'

# Ensure the boolean columns are converted to strings for Metadata compatibility
md = md.astype({col: 'str' for col in md.columns[md.dtypes == 'bool']})

# Create a new column 'category_label' for more descriptive labels
category_mapping = {
    'clear_zone': 'Acne Non-lesional',
    'acne': 'Acne Lesional',
    'control': 'Healthy'
}
md['category_label'] = md['category'].map(category_mapping)

# Create the 'lesional_score' column based on conditions
md['lesional_score'] = md.apply(
    lambda row: 'Low' if row['category_label'] == 'Acne Lesional' and row['local_lesion_severity'] in [1, 2] else
                'Moderate' if row['category_label'] == 'Acne Lesional' and row['local_lesion_severity'] in [3, 4] else
                'High' if row['category_label'] == 'Acne Lesional' and row['local_lesion_severity'] in [5, 6] else
                None, axis=1
)

# Save tsv
md.to_csv('../Metadata/57610_57610_analysis_mapping_category_label.csv')

md

,processing_robot,well_id_96,orig_name,platform,pcr_primers,tm50_8_tool,primer_date,run_prefix,extractionkit_lot,sample_plate,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,category,local_lesion_severity,category_label,lesional_score
#SampleID,,,,,,,,,,,,,,,,,,,,,
173620.14901.LAMI.RD308.D16.C1,Echo550_1,B4,LAMI.RD308.D16.C1,Illumina,FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT,409185Z,2/22/23,230425_M05314_0364_000000000-L4482_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV4_14901_Plate_1,...,0,0,33.7656381300,acne,RD308,acne,acne,4,Acne Lesional,Moderate
173620.14901.LAMI.RD310.D21.C1,Echo550_1,A12,LAMI.RD310.D21.C1,Illumina,FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT,409185Z,2/22/23,230425_M05314_0364_000000000-L4482_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV4_14901_Plate_2,...,36,72,31.9194777500,acne,RD310,acne,acne,4,Acne Lesional,Moderate
173620.14901.LAMI.RD305.D21.C3,Echo550_1,H10,LAMI.RD305.D21.C3,Illumina,FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT,409185Z,2/22/23,230425_M05314_0364_000000000-L4482_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV4_14901_Plate_4,...,26,69,22.1525032700,acne,RD305,healthy,clear_zone,0,Acne Non-lesional,None
173620.14901.LAMI.RD306.D18.C2,Echo550_1,D1,LAMI.RD306.D18.C2,Illumina,FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT,409185Z,2/22/23,230425_M05314_0364_000000000-L4482_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV4_14901_Plate_2,...,0,0,27.3979177600,acne,RD306,acne,acne,2,Acne Lesional,Low
173620.14901.LAMI.RD306.D7.C2,Echo550_1,E10,LAMI.RD306.D7.C2,Illumina,FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT,409185Z,2/22/23,230425_M05314_0364_000000000-L4482_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV4_14901_Plate_4,...,13,90,28.8194506500,acne,RD306,acne,acne,3,Acne Lesional,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173980.14901.LAMI.RD304.D28.C2,Echo550_1,G4,LAMI.RD304.D28.C2,Illumina,FWD:AGAGTTTGATCCTGGCTCAG; REV:TTACCGCGGCTGCTGGCA,409185Z,2/22/23,230504_M05314_0366_000000000-L446B_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV1-V3_14901_Plate_2,...,15,38,28.2191587900,acne,RD304,acne,acne,4,Acne Lesional,Moderate
173980.14901.LAMI.RD313.D14.C1,Echo550_1,G5,LAMI.RD313.D14.C1,Illumina,FWD:AGAGTTTGATCCTGGCTCAG; REV:TTACCGCGGCTGCTGGCA,409185Z,2/22/23,230504_M05314_0366_000000000-L446B_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV1-V3_14901_Plate_2,...,21,80,18.8717295500,acne,RD313,acne,acne,1,Acne Lesional,Low
173980.14901.LAMI.RD317.D9.C2,Echo550_1,F1,LAMI.RD317.D9.C2,Illumina,FWD:AGAGTTTGATCCTGGCTCAG; REV:TTACCGCGGCTGCTGGCA,409185Z,2/22/23,230504_M05314_0366_000000000-L446B_SMPL1_S1_L001,2201067R,CMI_LOreal_Acne_16SV1-V3_14901_Plate_3,...,0,0,22.5139432400,acne,RD317,acne,acne,2,Acne Lesional,Low


In [253]:
# Use 'category_label' in metadata for the rPCA analysis and plots
meta_v3 = md.copy()
meta_v4 = md.copy()

meta_v3['c_zone_group'] = meta_v3.cohort+'_'+meta_v3.c_zone
meta_v3['id_loc'] = meta_v3.subject_randomization_id+'_'+meta_v3.c_zone

meta_v4['c_zone_group'] = meta_v4.cohort+'_'+meta_v4.c_zone
meta_v4['id_loc'] = meta_v4.subject_randomization_id+'_'+meta_v4.c_zone

q2_meta_v3 = Metadata(meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x))
q2_meta_v4 = Metadata(meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x))

# Save the modified metadata as qiime2 Metadata for RPCA
metadata = Metadata.load('../Metadata/57610_57610_analysis_mapping.txt')

# Filter the tables
v3_table = Artifact.load('../Data/16S/Tables/16S_V1-V3_rarefied_uncollapsed_table.qza')
v4_table = Artifact.load('../Data/16S/Tables/16S_V4_rarefied_uncollapsed_table.qza')

# Extract the underlying biom.Table from the Artifact object
v3_biom_table = v3_table.view(biom.Table)
v4_biom_table = v4_table.view(biom.Table)

v3_table_filt = filter_samples(table=v3_biom_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)
v4_table_filt = filter_samples(table=v4_biom_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)

# Define colors and shapes for severity groups

# Set the color palette for the groups in the correct order
group_colors = {
    'Healthy': '#3333B3',     # Dark Blue color for Healthy
    'Acne Non-lesional': '#5cbccb',     # Blue color for Acne Non-Lesional
    'Acne Lesional': '#f16c52'       # Red color for Acne Lesional
}


# severity_order = ["absent Healthy", "absent Acne Non-lesional", "low Acne Non-lesional", "low Acne Lesional", "moderate Acne Lesional", "high Acne Lesional"]
group_order = ["Healthy", "Acne Non-lesional", "Acne Lesional", ]


# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt, q2_meta_v3, 'RPCA - 16S rRNA (V1-V3)', 'V1-V3', '../Figures/16S_Figures/rPCA/'),
    (v4_table_filt, q2_meta_v4, 'RPCA - 16S rRNA (V4)', 'V4', '../Figures/16S_Figures/rPCA/')
]

# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    # Add ellipse to the plot
    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)



/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/3250434935.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  q2_meta_v3 = Metadata(meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x))
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/3250434935.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  q2_meta_v4 = Metadata(meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x))


In [ ]:
# Define colors for severity levels
severity_colors = {
    'Low': '#F1948A',       # Light red for low severity
    'Moderate': '#EC7063',  # Red for moderate severity
    'High': '#C0392B'       # Dark red for high severity
}

# RPCA analysis and plotting
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    print(plot_title)
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance = rpca(table)

    # Save distance
    
    # Directly access the ordination data
    ordination = ordination_artifact
    
    plt.clf()

    # Extract sample ordinations and feature coordinates
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']
    spca_df["category_label"] = spca_df.index.map(metadata.to_dataframe()["category_label"])
    
    # Merge 'lesional_score' from 'md' to spca_df
    spca_df['lesional_score'] = spca_df.index.map(metadata.to_dataframe()['lesional_score'])

    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    
    
    # Convert metadata to a pandas DataFrame
    metadata_df = metadata.to_dataframe()

    # Find the intersection of IDs
    common_ids = set(metadata_df.index).intersection(distance.ids)

    # Filter metadata
    metadata_df = metadata_df.loc[list(common_ids)]

    # Filter distance matrix
    distance_matrix = distance.filter(common_ids, strict=True)

    group_column = 'category_label'  # Column in metadata defining the groups
    groups = metadata_df[group_column].unique()

    # Convert the distance object to a DistanceMatrix object if needed
    distance_matrix = DistanceMatrix(distance.to_data_frame().values, ids=distance.ids)

    # Store pairwise PERMANOVA results
    results = []

    # Perform pairwise PERMANOVA
    for group1, group2 in combinations(groups, 2):
        # Subset metadata and distance matrix for the two groups
        subset_ids = metadata_df[metadata_df[group_column].isin([group1, group2])].index
        subset_distance = distance_matrix.filter(subset_ids, strict=True)
        subset_metadata = metadata_df.loc[subset_ids]

        # Run PERMANOVA
        permanova_result = permanova(subset_distance, subset_metadata[group_column])
        results.append({
            'Group1': group1,
            'Group2': group2,
            'Test Statistic': permanova_result['test statistic'],
            'p-value': permanova_result['p-value']
            # 'Explained Variance (%)': permanova_result['explained variance'] * 100,
        })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)

    # Display results
    print(results_df)

    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(7, 10))
    
    # Scatter plot for each severity group with its specific color
    for group in group_order:
        if group in spca_df['category_label'].values:
            group_data = spca_df[spca_df['category_label'] == group]
            for group_name, data in group_data.groupby('category_label'):
                sns.scatterplot(
                    data=data,
                    x="PC1",
                    y="PC2",
                    facecolor=group_colors[group],
                    edgecolor='k',
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=group,
                    marker="o",
                    s=100
                )     

    # Draw ellipses based on category_label and add stars at centers for severity groups
    for category_label, data in spca_df.groupby("category_label"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)

        if len(data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define the color of the ellipse based on category_label
            ellipse_color = {
                "Healthy": "#423fa6",
                "Acne Non-lesional": "#67b2be",   
                "Acne Lesional": "#df7963"     
            }[category_label]

            # Draw the ellipse for each category_label
            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add stars based on severity_group
    for group, data in spca_df.groupby("category_label"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        # Use the color defined for each severity_group
        star_color = group_colors.get(group, "#999999")
        
        # Add a star at the mean location for each severity_group
        ax.scatter(mean[0], mean[1], color=star_color, marker=(8, 2, 0), s=300, edgecolor='black', zorder=5, linewidth=1)

    # Now, color the 'Acne Lesional' dots based on their 'lesional_score'
    lesional_data = spca_df[spca_df['category_label'] == 'Acne Lesional']
    
    # Only plot if the severity group is low, moderate, or high
    for severity, color in severity_colors.items():
        severity_data = lesional_data[lesional_data['lesional_score'] == severity]
        
        if len(severity_data) > 0:  # Check if there are any data points for this severity level
            sns.scatterplot(
                data=severity_data,
                x="PC1",
                y="PC2",
                facecolor=color,
                edgecolor='k',
                alpha=0.8,
                linewidth=0.5,
                ax=ax,
                label=f'{severity.capitalize()} Severity',
                marker="o",
                s=100
            )

    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=18)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=18)
    ax.tick_params(axis='both', which='major', labelsize=5)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=18)

    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=18)

    # Adjust plot style and save
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)

    # Set the thickness of the frame (axes spines)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    # Set plot title dynamically
    if plot_title == 'RPCA - 16S rRNA (V1-V3)':
        new_plot_title = '16S rRNA (V1-V3) rPCA Beta Diversity'
        plt.title(new_plot_title, fontsize=20)
    elif plot_title == 'RPCA - 16S rRNA (V4)':
        new_plot_title = '16S rRNA (V4) rPCA Beta Diversity'
        plt.title(new_plot_title, fontsize=20)    

    # Define custom handles and labels
    handles, labels = ax.get_legend_handles_labels()

    # Filter the handles and labels for the groups you want
    filtered_handles_labels = [(handle, label) for handle, label in zip(handles, labels) if label in ['Healthy', 'Acne Non-lesional', 'Acne Lesional']]  # Modify this list as needed

    # Set the legend with the filtered handles and labels
    ax.legend(
        handles=[handle for handle, label in filtered_handles_labels],
        labels=[label for handle, label in filtered_handles_labels],
        frameon=False,
        fontsize=18,
        title_fontsize=18,
        loc='best',
    )

    # Mapping for shorter names
    group_name_map = {
        "Healthy": "H",
        "Acne Non-lesional": "ANL",
        "Acne Lesional": "AL"
    }

    # Generate text with shortened names and significance stars
    pvalue_text = '\n'.join(
        [
            f"{group_name_map[row['Group1']]} vs {group_name_map[row['Group2']]}: "
            f"p={row['p-value']:.3f}"
            f"{' ***' if row['p-value'] <= 0.001 else ' **' if 0.001 < row['p-value'] <= 0.01 else ''}"
            for _, row in results_df.iterrows()
        ]
    )

    # Adjust x and y coordinates to place the text at the bottom-left corner
    ax.text(
        x=ax.get_xlim()[0] + 0.01,  # Slightly offset from the left axis
        y=ax.get_ylim()[0] + 0.01,  # Slightly offset from the bottom axis
        s=pvalue_text,
        fontsize=14,
        verticalalignment='bottom',
        horizontalalignment='left',
        bbox=dict(boxstyle="round,pad=0.3", edgecolor="gray", facecolor="white", alpha=0.8)
    )

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{file_suffix}_rPCA.png"), dpi=600)
    plt.close(fig)
    print(f"Plot saved to {output_path}")


RPCA - 16S rRNA (V1-V3)
              Group1             Group2  Test Statistic      p-value
0      Acne Lesional  Acne Non-lesional    1.8318518522 0.1600000000
1      Acne Lesional            Healthy    9.6296336567 0.0010000000
2  Acne Non-lesional            Healthy   12.3042027249 0.0010000000


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/701810104.py:151: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=18)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/701810104.py:152: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=18)


Plot saved to ../Figures/16S_Figures/rPCA/
RPCA - 16S rRNA (V4)
              Group1             Group2  Test Statistic      p-value
0      Acne Lesional  Acne Non-lesional    0.7116877407 0.5170000000
1      Acne Lesional            Healthy    8.4359278108 0.0010000000
2  Acne Non-lesional            Healthy    5.6372070633 0.0070000000


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/701810104.py:151: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=18)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_62437/701810104.py:152: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=18)


Plot saved to ../Figures/16S_Figures/rPCA/
